In [ ]:
import io
import os
import re
import requests
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [9]:
# connect database
import os
if os.path.exists('pass.env'):
    load_dotenv('pass.env')
else:
    load_dotenv('../pass.env')


db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

db_url = f"postgresql://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
engine = create_engine(db_url)

In [10]:
# Dynamic TTM (Trailing 12 Months) Top 10 States Query
verify_states_query = """
    WITH LatestData AS (
    SELECT MAX(order_purchase_timestamp) AS max_date 
    FROM orders
)
SELECT 
    c.customer_state,
    COUNT(DISTINCT c.customer_unique_id) AS unique_customers,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(CAST(SUM(oi.price) AS numeric), 2) AS total_revenue
FROM 
    customers c
JOIN 
    orders o ON c.customer_id = o.customer_id
JOIN 
    order_items oi ON o.order_id = oi.order_id
CROSS JOIN 
    LatestData ld 
WHERE 
    o.order_status = 'delivered'
    AND o.order_purchase_timestamp >= (ld.max_date - INTERVAL '1 year')
GROUP BY 
    c.customer_state
ORDER BY 
    total_orders DESC
LIMIT 10;
"""

In [11]:
with engine.connect() as connection:
    df_top_10_olist = pd.read_sql(text(verify_states_query), connection)

# 4. Display the results
print("-" * 70)
print(df_top_10_olist.to_string(index=False))
print("-" * 70)

----------------------------------------------------------------------
customer_state  unique_customers  total_orders  total_revenue
            SP             28558         29337     3643472.63
            RJ              8164          8389     1175101.52
            MG              7728          7918     1100151.17
            RS              3450          3529      492069.14
            PR              3364          3436      468899.88
            SC              2402          2446      352814.14
            BA              2159          2214      329875.26
            DF              1485          1524      213762.19
            GO              1328          1362      191901.22
            ES              1333          1361      191600.59
----------------------------------------------------------------------


In [12]:
target_states = ['SP', 'RJ', 'MG', 'RS', 'PR', 'SC', 'BA', 'DF', 'GO', 'ES']

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

In [ ]:
# Helper Functions 
def clean_metric(val):
    """The function removes junk text and HTML footnotes, leaving only the numbers"""
    if pd.isna(val): return None
    val = str(val)
    val = re.sub(r'\[.*?\]', '', val) 
    val = re.sub(r'[^\d.]', '', val.replace(',', '')) 
    try:
        return float(val)
    except ValueError:
        return None

def find_col(df, keyword):
    """column search function breaks down the MultiIndex barrier"""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(map(str, c)).strip() for c in df.columns]
        
    for c in df.columns:
        if keyword.lower() in str(c).lower():
            return c
    raise ValueError(f"Error: No columns containing the word were found '{keyword}' in this table")

In [ ]:

# Retrieving the latest metadata from the Brazilian government API.
print("Fetching official metadata from IBGE API...")
url_ibge = "https://servicodados.ibge.gov.br/api/v1/localidades/estados"
response = requests.get(url_ibge)
df_ibge = pd.DataFrame([{
    'state_code': s['sigla'],
    'state_name': s['nome']
} for s in response.json()])

In [ ]:
# Scan the Population, Area, and HDI data from the Wiki main page.
print("Scraping Population, Area, and HDI from Wikipedia Main Page...")
url_main = "https://en.wikipedia.org/wiki/Federative_units_of_Brazil"
res_main = requests.get(url_main, headers=headers)

tables_main = pd.read_html(io.StringIO(res_main.text), match="Capital")
df_main_raw = tables_main[0]

c_code = find_col(df_main_raw, 'Code')
c_pop = find_col(df_main_raw, 'Population')
c_area = find_col(df_main_raw, 'km2') 
c_hdi = find_col(df_main_raw, 'HDI')

df_main = df_main_raw[[c_code, c_pop, c_area, c_hdi]].copy()
df_main.rename(columns={c_code: 'state_code', c_pop: 'population', c_area: 'area_sq_km', c_hdi: 'hdi'}, inplace=True)

df_main['state_code'] = df_main['state_code'].astype(str).str.extract(r'([A-Z]{2})')

df_main['population'] = df_main['population'].apply(clean_metric)
df_main['area_sq_km'] = df_main['area_sq_km'].apply(clean_metric)
df_main['hdi'] = df_main['hdi'].apply(clean_metric)

In [ ]:
# GDP data from the Wiki's economics page.
print("Scraping GDP per capita from Wikipedia Economics Page...")
url_gdp = "https://en.wikipedia.org/wiki/List_of_Brazilian_federative_units_by_gross_domestic_product"
res_gdp = requests.get(url_gdp, headers=headers)
tables_gdp = pd.read_html(io.StringIO(res_gdp.text), match="capita")
df_gdp_raw = tables_gdp[0]

c_state = find_col(df_gdp_raw, 'Federative unit')
c_gdp = find_col(df_gdp_raw, 'capita')

df_gdp = df_gdp_raw[[c_state, c_gdp]].copy()
df_gdp.rename(columns={c_state: 'wiki_state_name', c_gdp: 'gdp_per_capita_brl'}, inplace=True)

df_gdp['wiki_state_name'] = df_gdp['wiki_state_name'].astype(str).apply(lambda x: re.sub(r'\[.*?\]', '', x).strip())
df_gdp['gdp_per_capita_brl'] = df_gdp['gdp_per_capita_brl'].apply(clean_metric)

In [ ]:
# Data Merging
print("4. Merging all sources dynamically...")

df_merged = pd.merge(df_ibge, df_main, on='state_code', how='inner')

df_merged = pd.merge(df_merged, df_gdp, left_on='state_name', right_on='wiki_state_name', how='inner')

df_merged['population_density'] = round(df_merged['population'] / df_merged['area_sq_km'], 2)

final_cols = ['state_code', 'state_name', 'population', 'area_sq_km', 'population_density', 'gdp_per_capita_brl', 'hdi']
df_final = df_merged[final_cols]
df_final = df_final[df_final['state_code'].isin(target_states)]

print("-" * 80)
print(df_final.to_string(index=False))
print("-" * 80)

In [ ]:
# Load to database
print("\nLoading dim_demographics to database...")
df_final.to_sql('dim_demographics', engine, if_exists='replace', index=False)
with engine.connect() as connection:
    connection.execute(text("ALTER TABLE dim_demographics ADD PRIMARY KEY (state_code);"))
    connection.commit()
    
print("Success! Multi-Source Zero-Hardcode Pipeline completed.")